## Purpose
This notebook performs all the data insertion and relationship setup for our Neo4J instance.

Explanation of classes used:
* CypherQueryLoader: class to load in our cypher queries needed to insert data. These cypher queries are stored in .cql files located within the queries directory
* FBRefDataLoader: needed to load in the fixtures, players and teams json files for insertion
* FBRefDataFormatter: needed to split up our match json file into 3 files:
1. General Match data: time, place, venue, referee, home, away etc
2. Players performance during the match
3. Goalie performance during the match - this is not part of players performance during the match as (well this is obvious, he wont really have tackles/shots on target etc)

Psuedo for code:
1. Load in our environment variables that we setup previously, this will allow us to conncet to our neo4j instance.
2. Load in our utility functions and setup the neo4j driver
3. Load in our cypher queries
4. Now for each season that we have within our data directory:
    1. load in match, team and player json's
    2. Format the data
    3. run our insertion queries. Session.run needs to know a) the query and b) the data for the query so we pass it parameters. self explanatory

In [11]:
import os
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
from dotenv import load_dotenv 
from utilities.cypher_query_loader import CypherQueryLoader
from utilities.fbref import FBRefDataLoader, FBRefDataFormatter
import json
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

# dylan instance details 
load_dotenv(dotenv_path = os.path.join(os.getcwd(),'config/.env'))

NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")

CQL = CypherQueryLoader(os.path.join(os.getcwd(), 'queries/'))
DataLoader = FBRefDataLoader(os.path.join(os.getcwd(),'data/'))
DataFormatter = FBRefDataFormatter()

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

gameweeks_season_query = CQL.load_query("season_gameweeks_creation")
player_creation_query = CQL.load_query("player_creation")
match_creation_query = CQL.load_query("match_creation")
player_gamestats_query = CQL.load_query("player_game_stats")
goalie_gamestats_query = CQL.load_query("goalie_game_stats")

data_directory = os.path.join(os.getcwd(),'data')
for season_directory in DataLoader.listdir(data_directory):
    print(season_directory)
    
    match_reports = DataLoader.load_data("fixtures", season_directory)
    team_details = DataLoader.load_data("teams", season_directory)
    player_details = DataLoader.load_data("players", season_directory)

    player_details = DataFormatter.add_team_name_to_player(player_details, team_details)
    game_weeks_data = [{"gw_number": f"GW{gw}", "season_name": season_directory} for gw in range(1, 39)]
    match_data, player_stats_data, goalkeeper_stats_data = DataFormatter.split_match_data(match_reports)

    # Execute Queries
    with driver.session(database="neo4j") as session:
        session.run(gameweeks_season_query, parameters={"game_weeks": game_weeks_data})
        session.run(player_creation_query, parameters={"players": player_details, "season_name": season_directory})
        session.run(match_creation_query, parameters={"matches": match_data, "season_name": season_directory})
        session.run(player_gamestats_query, parameters={
            "player_stats": player_stats_data,
            "season_name": season_directory
        })
        session.run(goalie_gamestats_query, parameters={
            "goalkeeper_stats": goalkeeper_stats_data,
            "season_name": season_directory
        })

2021-2022


ClientError: {code: Neo.ClientError.Transaction.TransactionHookFailed} {message: You have exceeded the logical size limit of 400000 relationships in your database (attempt to add 1900 relationships would reach 401070 relationships). Please consider upgrading to the next tier.}